In [1]:
%pip install -qU langgraph langchain langchain-groq langchain-tavily python-dotenv aiosqlite langgraph-checkpoint-sqlite

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

# Cargamos las variables de entorno desde tu archivo .env
load_dotenv()

# Seteamos las llaves correctas para tu pipeline con Groq
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [3]:
import operator
import sqlite3
from dataclasses import dataclass, field
from typing import Annotated, Any, Dict, List

from langchain_core.messages import (
    AnyMessage,
    BaseMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)
# 🚀 CORRECCIÓN: Cambiamos Gemini por Groq
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, StateGraph
from typing_extensions import TypedDict

In [4]:
import operator
import sqlite3
from typing import Annotated
from langchain_core.messages import AnyMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from typing_extensions import TypedDict


# 1. Definimos el estado del Agente (Grafo)
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]


# 2. Configuramos la persistencia local con SQLite
conn = sqlite3.connect("checkpoints.db", check_same_thread=False)
memory = SqliteSaver(conn)

In [5]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        
        graph = StateGraph(AgentState)

        # 🚀 CAMBIO: Ahora el nodo apunta formalmente a self.call_groq
        graph.add_node("llm", self.call_groq)
        graph.add_node("action", self.take_action)

        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")

        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    # 🚀 CAMBIO: Renombrado el método a call_groq
    def call_groq(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages

        print("--- 🧠 LLM (Groq) Pensando ---")
        print("Mensajes enviados al modelo:", messages)
        
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        
        for t in tool_calls:
            print(f"🚀 Ejecutando Herramienta: '{t['name']}'")
            print(f"   Argumentos pasados por Groq: {t['args']}")
            
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
            
        print("🔄 Retornando el resultado al flujo de la LLM...")
        return {'messages': results}

In [12]:
import os
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch

# 1. Validación de la API Key de Tavily
if not TAVILY_API_KEY:
    raise ValueError(
        "TAVILY_API_KEY no encontrada. Asegúrate de que esté en tu archivo .env."
    )

# 2. Inicialización de la Herramienta de Búsqueda (Tavily)
tool = TavilySearch(max_results=3, tavily_api_key=TAVILY_API_KEY)

# 3. Definición del Prompt de Sistema del Agente

# Ejercicio
# prompt_system = """Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.
# Tienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).
# Busca información únicamente cuando tengas certeza de qué buscar.
# Si necesitas más detalles para formular una pregunta de seguimiento, tienes permiso para hacerlo.
# Cuando se te solicite comparar información (por ejemplo: cuál es más caliente, más grande, etc.), utiliza la información del historial de la conversación y los resultados de las herramientas."""

# Corregido
prompt_system = """Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.
Tienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).
Busca información únicamente cuando tengas certeza de qué buscar.
Cuando busques datos en tiempo real como el clima, asegúrate de realizar búsquedas generales y no restringidas a noticias."""

# 🚀 CORRECCIÓN: Usamos el modelo ultra rápido de Groq con temperatura 0 para respuestas precisas
model_groq = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

# 4. Instanciación final del Agente Cíclico con Memoria
abot = Agent(model_groq, [tool], system=prompt_system, checkpointer=memory)

In [15]:
from langchain_core.messages import HumanMessage

# 1. Definimos la nueva pregunta para Kansas City
messages = [HumanMessage(content="¿Cómo está el clima en Kansas City - USA hoy?")]

# 2. Configuramos el ID del hilo de la conversación para la memoria SQLite
thread = {"configurable": {"thread_id": "22"}}

print("\n--- Pregunta 1: Clima en Kansas City ---")

# 3. Ejecutamos el flujo del grafo en streaming
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        # Capturamos dinámicamente los estados del grafo
        if k in ("llm", "action", "call_groq", "take_action"): 
            print(f"[{k}]: {v['messages']}")


--- Pregunta 1: Clima en Kansas City ---
--- 🧠 LLM (Groq) Pensando ---
Mensajes enviados al modelo: [SystemMessage(content='Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nCuando busques datos en tiempo real como el clima, asegúrate de realizar búsquedas generales y no restringidas a noticias.', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Kansas City - USA hoy?', additional_kwargs={}, response_metadata={})]
[llm]: [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'naaz57dcy', 'function': {'arguments': '{"query":"Kansas City USA weather today","time_range":"day","topic":"general"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_token

In [16]:
from langchain_core.messages import HumanMessage

# 1. Definimos la nueva pregunta para Kansas City
messages = [HumanMessage(content="¿Y el clima en Buenos Aires - Argentina hoy?")]

# 2. Configuramos el ID del hilo de la conversación para la memoria SQLite
thread = {"configurable": {"thread_id": "22"}}

print("\n--- Pregunta 1: Clima en  Buenos Aires - Argentina ---")

# 3. Ejecutamos el flujo del grafo en streaming
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        # Capturamos dinámicamente los estados del grafo
        if k in ("llm", "action", "call_groq", "take_action"): 
            print(f"[{k}]: {v['messages']}")


--- Pregunta 1: Clima en  Buenos Aires - Argentina ---
--- 🧠 LLM (Groq) Pensando ---
Mensajes enviados al modelo: [SystemMessage(content='Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nCuando busques datos en tiempo real como el clima, asegúrate de realizar búsquedas generales y no restringidas a noticias.', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Kansas City - USA hoy?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'naaz57dcy', 'function': {'arguments': '{"query":"Kansas City USA weather today","time_range":"day","topic":"general"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion

In [17]:
from langchain_core.messages import HumanMessage

# 1. Definimos la pregunta de comparación (depende de la memoria del chat)
messages = [HumanMessage(content="¿Cuál ciudad está más caliente?")]

# 2. Mantenemos el thread_id = "3" para que lea el historial de SQLite
thread = {"configurable": {"thread_id": "22"}}

print("\n--- Pregunta 3: Comparación ---")

# 3. Corremos el flujo en streaming capturando los nodos de tu grafo de Groq
for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        if k in ("llm", "action", "call_groq", "take_action"):
            print(f"[{k}]: {v['messages']}")


--- Pregunta 3: Comparación ---
--- 🧠 LLM (Groq) Pensando ---
Mensajes enviados al modelo: [SystemMessage(content='Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nCuando busques datos en tiempo real como el clima, asegúrate de realizar búsquedas generales y no restringidas a noticias.', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Kansas City - USA hoy?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'naaz57dcy', 'function': {'arguments': '{"query":"Kansas City USA weather today","time_range":"day","topic":"general"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_t

In [18]:
print("\n--- Pergunta 4: Comparación ---")

messages = [HumanMessage(content="¿Cuál ciudad está más caliente de las dos?")]
thread = {"configurable": {"thread_id": "22"}}  # O el ID del hilo donde guardaste AMBAS búsquedas

for event in abot.graph.stream({"messages": messages}, thread):
    for k, v in event.items():
        # Filtramos por los nombres reales de tus nodos (call_groq y take_action)
        if k in ("call_groq", "take_action"):
            # Usamos [-1] para imprimir solo el último mensaje generado en ese nodo
            print(f"[{k}]: {v['messages'][-1].content}")


--- Pergunta 4: Comparación ---
--- 🧠 LLM (Groq) Pensando ---
Mensajes enviados al modelo: [SystemMessage(content='Eres un asistente de investigación inteligente. Usa el motor de búsqueda (tavily_search_results_json) para buscar información.\nTienes permiso para realizar múltiples llamadas a la herramienta (de forma conjunta o en secuencia).\nBusca información únicamente cuando tengas certeza de qué buscar.\nCuando busques datos en tiempo real como el clima, asegúrate de realizar búsquedas generales y no restringidas a noticias.', additional_kwargs={}, response_metadata={}), HumanMessage(content='¿Cómo está el clima en Kansas City - USA hoy?', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'naaz57dcy', 'function': {'arguments': '{"query":"Kansas City USA weather today","time_range":"day","topic":"general"}', 'name': 'tavily_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_t